In [1]:
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus
import os

In [3]:
# 1. LOAD ENVIRONMENT VARIABLES

load_dotenv()

mysql_user = os.getenv("MYSQL_USER")
mysql_password = quote_plus(os.getenv("MYSQL_PASSWORD"))
mysql_host = os.getenv("MYSQL_HOST")
mysql_port = os.getenv("MYSQL_PORT")
mysql_database = os.getenv("MYSQL_DATABASE")


In [4]:

# 2. CREATE MYSQL CONNECTION

try:
    engine = create_engine(
        f"mysql+pymysql://{mysql_user}:{mysql_password}@"
        f"{mysql_host}:{mysql_port}/{mysql_database}"
    )

    # Test Connection
    with engine.connect() as connection:
        print("Successfully connected to MYSQL !")

except Exception as e:
    print(f"ERROR: Could not connect to MYSQL: {e}")


Successfully connected to MYSQL !


In [5]:
# ==============================================
# 3. READ BRONZE TABLE
# ==============================================

df = pd.read_sql(
    "SELECT * FROM bronze_sales",
    engine
)


In [6]:

# 4. VERIFY DATA

print("Bronze Data loaded successfully !!")

print("Rows:", len(df))

print("Columns:", len(df.columns))

print("\nFirst 5 rows:")
print(df.head())

Bronze Data loaded successfully !!
Rows: 9800
Columns: 21

First 5 rows:
   row_id        order_id  order_date   ship_date       ship_mode customer_id  \
0       1  CA-2017-152156  08/11/2017  11/11/2017    Second Class    CG-12520   
1       2  CA-2017-152156  08/11/2017  11/11/2017    Second Class    CG-12520   
2       3  CA-2017-138688  12/06/2017  16/06/2017    Second Class    DV-13045   
3       4  US-2016-108966  11/10/2016  18/10/2016  Standard Class    SO-20335   
4       5  US-2016-108966  11/10/2016  18/10/2016  Standard Class    SO-20335   

     customer_name    segment        country             city  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  postal_code regio

In [7]:
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nUnique values:")
print(df.nunique())

print("\nSales statistics:")
print(df["sales"].describe())


Data types:
row_id                          int64
order_id                       object
order_date                     object
ship_date                      object
ship_mode                      object
customer_id                    object
customer_name                  object
segment                        object
country                        object
city                           object
state                          object
postal_code                    object
region                         object
product_id                     object
category                       object
sub_category                   object
product_name                   object
sales                         float64
ingestion_timestamp    datetime64[ns]
source_file_name               object
load_id                        object
dtype: object

Missing values:
row_id                  0
order_id                0
order_date              0
ship_date               0
ship_mode               0
customer_id             0
cu

In [8]:
# 5. DATA QUALITY CHECK

# date validation

order_dates = pd.to_datetime(
    df["order_date"],
    errors = "coerce",
    dayfirst = True
)

print("Invalid Order Dates:", order_dates.isna().sum())

Invalid Order Dates: 0


In [9]:
ship_dates = pd.to_datetime(
    df["ship_date"],
    errors = "coerce",
    dayfirst = True
)

print("Invalid Ship Dates:", ship_dates.isna().sum())

Invalid Ship Dates: 0


In [10]:
# 6. CHECK DATE LOGIC

In [11]:
invalid_ship_dates = (ship_dates < order_dates).sum()
print(
    "Ship Date Before Order Dates:",
    invalid_ship_dates
)

Ship Date Before Order Dates: 0


In [12]:
# 7. Sales Quality Check

print("Negative Sales:", (df["sales"] < 0).sum())
print("Zero Sales:", (df["sales"] == 0).sum())

Negative Sales: 0
Zero Sales: 0


In [13]:
print("Top 10 Sales Values:")

print(
    df[["order_id","product_name","sales"]]
    .sort_values("sales", ascending=False)
    .head(10)
)

Top 10 Sales Values:
            order_id                                       product_name  \
2697  CA-2015-145317  Cisco TelePresence System EX90 Videoconferenci...   
6826  CA-2017-118689              Canon imageCLASS 2200 Advanced Copier   
8153  CA-2018-140151              Canon imageCLASS 2200 Advanced Copier   
2623  CA-2018-127180              Canon imageCLASS 2200 Advanced Copier   
4190  CA-2018-166709              Canon imageCLASS 2200 Advanced Copier   
9039  CA-2017-117121   GBC Ibimaster 500 Manual ProClick Binding System   
4098  CA-2015-116904               Ibico EPK-21 Electric Binding System   
4277  US-2017-107440   3D Systems Cube Printer, 2nd Generation, Magenta   
8488  CA-2017-158841  HP Designjet T520 Inkjet Large Format Printer ...   
6425  CA-2017-143714              Canon imageCLASS 2200 Advanced Copier   

          sales  
2697  22638.480  
6826  17499.950  
8153  13999.960  
2623  11199.968  
4190  10499.970  
9039   9892.740  
4098   9449.950  
4277   90

In [14]:
# ==========================================
# CATEGORY QUALITY CHECK
# ==========================================

print("\nShip Modes:")
print(df["ship_mode"].value_counts(dropna=False))

print("\nSegments:")
print(df["segment"].value_counts(dropna=False))

print("\nRegions:")
print(df["region"].value_counts(dropna=False))

print("\nCategories:")
print(df["category"].value_counts(dropna=False))

print("\nSub-Categories:")
print(df["sub_category"].value_counts(dropna=False))

print("\nCountry:")
print(df["country"].value_counts(dropna=False))

print("\nCity:")
print(df["city"].value_counts(dropna=False))

print("\nState:")
print(df["state"].value_counts(dropna=False))


Ship Modes:
ship_mode
Standard Class    5859
Second Class      1902
First Class       1501
Same Day           538
Name: count, dtype: int64

Segments:
segment
Consumer       5101
Corporate      2953
Home Office    1746
Name: count, dtype: int64

Regions:
region
West       3140
East       2785
Central    2277
South      1598
Name: count, dtype: int64

Categories:
category
Office Supplies    5909
Furniture          2078
Technology         1813
Name: count, dtype: int64

Sub-Categories:
sub_category
Binders        1492
Paper          1338
Furnishings     931
Phones          876
Storage         832
Art             785
Accessories     756
Chairs          607
Appliances      459
Labels          357
Tables          314
Envelopes       248
Bookcases       226
Fasteners       214
Supplies        184
Machines        115
Copiers          66
Name: count, dtype: int64

Country:
country
United States    9800
Name: count, dtype: int64

City:
city
New York City    891
Los Angeles      728
Philadelphi

In [15]:
# ==========================================
# WHITESPACE CHECK
# ==========================================

string_columns = df.select_dtypes(include="object").columns

for col in string_columns:
    whitespace_count = (
        df[col].astype(str).str.strip() != df[col].astype(str)
    ).sum()

    if whitespace_count > 0:
        print(
            f"{col}: {whitespace_count} values "
            "with leading/trailing whitespace"
        )

In [16]:
# Missing Postal Code Records

print("\nRows  with missing postal code:")

print(
    df[df["postal_code"].isna()][
        ["row_id", "city", "state", "postal_code"]
    ]
)


Rows  with missing postal code:
      row_id        city    state postal_code
2234    2235  Burlington  Vermont        None
5274    5275  Burlington  Vermont        None
8798    8799  Burlington  Vermont        None
9146    9147  Burlington  Vermont        None
9147    9148  Burlington  Vermont        None
9148    9149  Burlington  Vermont        None
9386    9387  Burlington  Vermont        None
9387    9388  Burlington  Vermont        None
9388    9389  Burlington  Vermont        None
9389    9390  Burlington  Vermont        None
9741    9742  Burlington  Vermont        None


In [17]:
# Check Duplicate Row Id

print("\nDuplicate row_id:")
print(df["row_id"].duplicated().sum())

print("\nRow ID range:")
print(df["row_id"].min(), "to", df["row_id"].max())


Duplicate row_id:
0

Row ID range:
1 to 9800
